# Context Engineering - A Hands-On Python Guide

Welcome, coder! This notebook teaches the fundamentals of **Context Engineering** — the discipline of designing the input context that controls how Large Language Models (LLMs) behave.

## What you'll learn:
- The 6 context layers that make up a prompt
- How to build and assemble prompts programmatically
- Use case presets (Customer Support, Code Review, etc.)
- Token estimation and context windows
- The "Lost in the Middle" effect
- Provider models and pricing
- **Real LLM calls with Groq API (free, no cost)**
- LLM response quality evaluation

---
## 1. Imports & Helper Functions

Let's set up the utilities we'll use throughout the notebook.

In [ ]:
import json
import random
import os

GROQ_API_KEY = os.environ.get("GROQ_API_KEY", "")

PRICING = {
    "openai": {
        "gpt-4o": {"input": 0.00250, "output": 0.01000},
        "gpt-4o-mini": {"input": 0.00015, "output": 0.00060},
    },
    "gemini": {
        "gemini-2.0-flash": {"input": 0.00010, "output": 0.00040},
    },
    "anthropic": {
        "claude-sonnet-4-20250514": {"input": 0.00300, "output": 0.01500},
    },
    "groq": {
        "llama-3.3-70b-versatile": {"input": 0.0, "output": 0.0},
        "mixtral-8x7b-32768": {"input": 0.0, "output": 0.0},
        "gemma2-9b-it": {"input": 0.0, "output": 0.0},
    }
}

def estimate_tokens(text):
    return len(text) // 4 + 1

def estimate_cost(provider, model, input_tokens, output_tokens):
    rates = PRICING.get(provider, {}).get(model, {})
    input_cost = (input_tokens / 1000) * rates.get("input", 0)
    output_cost = (output_tokens / 1000) * rates.get("output", 0)
    return round(input_cost + output_cost, 6)

def build_prompt(layers):
    parts = []
    for key, layer in layers.items():
        if layer["enabled"] and layer["content"]:
            parts.append(f"=== {layer['label']} ===")
            parts.append(layer["content"])
    return "\n\n".join(parts)

def show_layers(layers):
    for key, layer in layers.items():
        status = "ON" if layer["enabled"] else "OFF"
        content = layer["content"][:70].replace("\n", " | ")
        print(f"  [{status}] {layer['label']:20s} {content}...")

---
## 2. The 6 Context Layers

Think of an LLM like an operating system. Context is its `RAM` — everything loaded into the conversation window. There are 6 layers:

| # | Layer | Purpose |
|---|-------|---------|
| 1 | **System Prompt** | Sets the AI's persona, rules, and constraints |
| 2 | **User Input** | The actual query or task from the user |
| 3 | **Conversation History** | Past exchanges for continuity |
| 4 | **RAG Knowledge** | Retrieved documents (Retrieval-Augmented Generation) |
| 5 | **Recent Conversation** | The last few messages for immediate context |
| 6 | **State & Memory** | User preferences, session data, saved state |

In [ ]:
context_layers = {
    "system_prompt": {
        "label": "System Prompt",
        "content": "You are a helpful coding assistant. Be concise and clear.",
        "enabled": True
    },
    "user_input": {
        "label": "User Input",
        "content": "Explain the difference between lists and tuples in Python.",
        "enabled": True
    },
    "conversation_history": {
        "label": "Conversation History",
        "content": "User: What is Python?\nAssistant: Python is a high-level programming language.",
        "enabled": False
    },
    "rag_knowledge": {
        "label": "RAG Knowledge",
        "content": "Lists are mutable, ordered collections. Tuples are immutable, ordered collections.",
        "enabled": True
    },
    "recent_conversation": {
        "label": "Recent Conversation",
        "content": "",
        "enabled": False
    },
    "state_memory": {
        "label": "State & Memory",
        "content": "User skill level: beginner. Preferred language: Python.",
        "enabled": True
    }
}

print("Context Layers Loaded:")
show_layers(context_layers)

You can **enable/disable** layers to control exactly what the LLM sees. Disabled layers are excluded from the final prompt.

---
## 3. Building a Compound Prompt

The final prompt sent to the LLM is the **assembly of all enabled layers**.

In [ ]:
prompt = build_prompt(context_layers)
print("=== ASSEMBLED PROMPT ===")
print(prompt)
print("\n" + "=" * 40)
print(f"Words: {len(prompt.split())}  |  Est. Tokens: {estimate_tokens(prompt)}")

---
## 4. Use Case Presets

The Context Engineering Lab ships with 4 realistic presets. Each preset pre-fills all 6 layers with domain-specific content. Click one and run the cell below to load it.

In [ ]:
USE_CASE_PRESETS = {
    "customer_support": {
        "system_prompt": "You are a professional customer support agent for ShopEase.\n\nBe empathetic, polite, and solution-oriented.\n\nFollow company policies at all times.\n\nNever make promises outside the stated policy.\n\nAlways verify customer details before processing requests.",
        "user_input": "I ordered a pair of running shoes (order #SH-98472) 10 days ago but they arrived with a torn seam. I want a full refund or replacement. This is the second defective product I have received from your store this month.",
        "conversation_history": "User: What is your return policy for defective items?\nAgent: We accept returns within 30 days of delivery for defective items. You can get a full refund or replacement.\nUser: I have a defective product I received today.",
        "rag_knowledge": "Company Return Policy (v3.2):\n- Defective items: Full refund or replacement within 30 days\n- Refunds processed within 5-7 business days\n- Customer must provide photo evidence\n- Repeat defect cases (>1 in 6 months): Escalate to senior support\n- VIP customers (Tier 2+): Priority replacement shipping\n\nProduct Info: Shoe Model \"SwiftRun Pro\" - known defect rate: 0.3%",
        "recent_conversation": "Agent: I am sorry to hear about the defect. Let me look up your order SH-98472.\nAgent: I can see this is the second defective item for you this month. I will escalate this to our senior team.\nUser: I really want a smooth resolution this time.",
        "state_memory": "Customer Profile:\n- Name: Sarah Chen\n- Membership: VIP Tier 2 (Gold)\n- Joined: 14 months ago\n- Total Orders: 23\n- Previous Returns: 2 (both resolved)\n- Preferred Contact: Email\n- Language: English\n\nSession: Support ticket #TK-88231 | Open 12m"
    },
    "code_review": {
        "system_prompt": "You are a senior Python code reviewer.\n\nAnalyse code for bugs, performance issues, and style violations.\n\nProvide actionable, specific feedback.\n\nFollow PEP 8 and modern Python best practices.\n\nSuggest concrete fixes with code examples.",
        "user_input": "def process_data(items, threshold):\n    result = []\n    for i in range(len(items)):\n        if items[i] > threshold:\n            result.append(items[i] * 2)\n        else:\n            result.append(items[i])\n    return sorted(result, reverse=True)[:10]",
        "conversation_history": "Author: I wrote this function to filter and sort data but it is slow for large lists.\nReviewer: The loop pattern can be optimised. Let me look at the full context.\nAuthor: It takes ~3 seconds for a list of 100k items.",
        "rag_knowledge": "PEP 8 Style Guide:\n- Use list comprehensions over map/filter\n- Maximum line length: 79 characters\n- Use descriptive variable names\n\nProject Conventions:\n- Type hints required for all public functions\n- Use pathlib over os.path\n\nTeam Standards v2.1:\n- All PRs require 2 approvals\n- Unit tests mandatory for new functions",
        "recent_conversation": "Reviewer: The main concern here is O(n log n) sorting on every call.\nAuthor: Would memoization help?\nReviewer: Possibly, but algorithmic improvement is better.",
        "state_memory": "Project: DataPipeline v2.4\nRepository: github.com/company/data-pipeline\nBranch: feature/optimise-processing\n\nDeveloper: Alex M. (Mid-level, 2 yrs at company)\nPriority: Medium | Deadline: Next sprint (Fri)"
    },
    "medical_triage": {
        "system_prompt": "You are a medical triage assistant.\n\nIMPORTANT: You are NOT a doctor. This is not medical advice.\n\nAlways include a disclaimer to seek professional medical help for emergencies.\n\nUse the provided medical guidelines to assess symptom urgency.\n\nBe thorough but clear. Err on the side of caution.",
        "user_input": "I have had a persistent headache for 3 days. It is on the right side of my head, behind my eye. I also feel nauseous and am sensitive to light. I have never had migraines before. I am 34 years old, generally healthy.",
        "conversation_history": "Patient: I have been having headaches.\nNurse: When did they start?\nPatient: About 3 days ago.\nNurse: Any other symptoms like fever or vision changes?",
        "rag_knowledge": "Clinical Triage Guidelines (v5.1):\n\nRed Flag Symptoms (Seek Emergency):\n- Sudden severe thunderclap headache\n- Headache with fever and neck stiffness\n- Headache after head injury\n- New neurological symptoms\n\nMigraine Diagnostic Criteria (ICHD-3):\n- Headache lasting 4-72 hours\n- Unilateral location\n- Nausea/vomiting\n- Photophobia/phonophobia",
        "recent_conversation": "Nurse: Are you taking any medication?\nPatient: I tried ibuprofen but it does not help much.\nNurse: Any family history of migraines?\nPatient: My mother gets them.",
        "state_memory": "Patient Profile:\n- Age: 34\n- Sex: Female\n- Known Conditions: None\n- Allergies: Penicillin\n- Medications: None regular\n\nSession: Triage chat initiated | Duration: 8m\nUrgency Assessment: Non-emergent (scheduled consult)"
    },
    "creative_writing": {
        "system_prompt": "You are an experienced creative writing coach and editor.\n\nProvide constructive, encouraging feedback on creative writing.\n\nAnalyse narrative structure, character development, pacing, and prose style.\n\nSuggest improvements while preserving the writer's unique voice.\n\nReference literary techniques and devices where relevant.",
        "user_input": "CHAPTER 1: THE LAST TRAIN\n\nElara stepped onto the platform as the 9:15 pulled away, its headlight dissolving into the fog like a dying star. She had missed it. Again.\n\nHer phone buzzed - Mother. She let it ring.\n\nThe station clock read 9:17. The next train was at 5:47 AM. Seven hours in this godforsaken waiting room with its flickering fluorescent lights and the smell of stale coffee.\n\n\"Rough night?\"\n\nShe turned. A man in a weathered coat stood by the ticket machine, holding a paper cup. Steam curled from it like a question mark.",
        "conversation_history": "Writer: This is the opening chapter of my novel \"The Distance Between Us.\"\nEditor: The atmosphere is strong. Let us focus on pacing in the first draft.\nWriter: I am worried the opening is too slow.\nEditor: Slow openings work if the prose carries it.",
        "rag_knowledge": "Literary Techniques Reference:\n- In medias res: Starting in the middle of action\n- Showing vs Telling: Use sensory details over exposition\n- Chekhov's Gun: Every element should be necessary\n- Pacing: Short sentences = tension, long sentences = reflection\n\nGenre: Literary Fiction / Contemporary Drama\nTone: Melancholic, atmospheric, introspective",
        "recent_conversation": "Editor: The imagery is evocative. \"Steam curled like a question mark\" - excellent.\nWriter: Thank you. I want the stranger to be a catalyst.\nEditor: Good instinct. Introduce the central conflict through interaction, not exposition.",
        "state_memory": "Project: Novel \"The Distance Between Us\"\nAuthor: Maya R. (First-time novelist)\nStage: First draft (Chapter 1/20)\n\nLast Session Feedback:\n- Strong atmospheric prose\n- Dialogue needs more subtext\n- Show don't tell the protagonist's loneliness\n\nDeadline: Beta readers in 6 weeks"
    }
}

def load_preset(name):
    preset = USE_CASE_PRESETS[name]
    for key in context_layers:
        if key in preset:
            context_layers[key]["content"] = preset[key]
            context_layers[key]["enabled"] = True
    prompt = build_prompt(context_layers)
    print(f"Loaded preset: {name.replace('_', ' ').title()}")
    print(f"Prompt: {estimate_tokens(prompt)} tokens, {len(prompt.split())} words")
    return prompt

print("Available presets:", list(USE_CASE_PRESETS.keys()))
print("\nRun: load_preset('customer_support') to load a use case")

In [ ]:
# Pick one: customer_support, code_review, medical_triage, creative_writing
prompt = load_preset("customer_support")

In [ ]:
print("Current context layers:\n")
show_layers(context_layers)
print("\n" + "=" * 50)
print(prompt[:500] + "...")

---
## 5. Token Estimation & Context Windows

LLMs don't read characters — they read **tokens** (~4 chars each). Each model has a **context window**:

In [ ]:
models = [
    ("GPT-4o", 128000),
    ("Claude Sonnet 4", 256000),
    ("Gemini 2.0 Flash", 1048576),
    ("Llama 3.3 70B (Groq)", 32768),
]

tokens = estimate_tokens(prompt)
print(f"Your prompt: {tokens} tokens (words: {len(prompt.split())})")
print()
for name, window in models:
    pct = (tokens / window) * 100
    bar = "#" * int(pct / 2) + " " * (50 - int(pct / 2))
    print(f"  {name:25s} {bar} {pct:.2f}%")

> **Context rot:** If your prompt exceeds the window, the LLM truncates the oldest parts. The middle of your prompt gets the least attention (**Lost in the Middle** effect).

---
## 6. Provider Models & Pricing

Here's what each API call costs (pricing from the Context Engineering Lab project):

In [ ]:
input_tok = estimate_tokens(prompt)
output_tok = 200

print(f"Estimated cost for {input_tok} input + {output_tok} output tokens:\n")
for provider, models in PRICING.items():
    for model in models:
        cost = estimate_cost(provider, model, input_tok, output_tok)
        print(f"  {provider:12s} {model:30s} ${cost:.6f}")

---
## 7. Real LLM Calls with Groq API

Groq provides **free** access to Llama 3.3, Mixtral, and Gemma models. Let's use their OpenAI-compatible API to make real calls.

> **Get a free API key:** https://console.groq.com/keys

In [ ]:
if not GROQ_API_KEY:
    try:
        GROQ_API_KEY = input("Enter your Groq API key: ").strip()
    except EOFError:
        pass

print(f"API Key: {'[SET]' if GROQ_API_KEY else '[MISSING]'}")
print("Get a free key at: https://console.groq.com/keys")
print("Or set: export GROQ_API_KEY='gsk_...'")

In [ ]:
groq_ready = False
GROQ_MODELS = ["llama-3.3-70b-versatile", "mixtral-8x7b-32768", "gemma2-9b-it"]

if GROQ_API_KEY:
    try:
        from openai import OpenAI
        client = OpenAI(api_key=GROQ_API_KEY, base_url="https://api.groq.com/openai/v1")
        groq_ready = True
        print("Groq client ready!")
        print(f"Available models: {GROQ_MODELS}")
    except ImportError:
        print("Install openai package: pip install openai")
else:
    print("Skipping - no API key provided.")

In [ ]:
def groq_generate(layers, model="llama-3.3-70b-versatile", temperature=0.7, max_tokens=1024):
    prompt = build_prompt(layers)
    print(f"Sending {estimate_tokens(prompt)} tokens to {model}...")
    
    response = client.chat.completions.create(
        model=model,
        messages=[
            {"role": "system", "content": layers["system_prompt"]["content"]},
            {"role": "user", "content": prompt}
        ],
        temperature=temperature,
        max_tokens=max_tokens,
    )
    
    content = response.choices[0].message.content
    usage = response.usage
    
    print(f"Response: {len(content.split())} words, {usage.completion_tokens} tokens")
    print(f"Total tokens used: {usage.total_tokens} (FREE)\n")
    return content

# Run this cell to call Groq
if groq_ready:
    result = groq_generate(context_layers)
    print("=== LLM RESPONSE ===")
    print(result)
else:
    print("Run the Groq setup cell first.")

In [ ]:
# Try a different model
if groq_ready:
    result = groq_generate(context_layers, model="mixtral-8x7b-32768", temperature=0.3)
    print("=== MIXTRAL RESPONSE ===")
    print(result)
else:
    print("Run the Groq setup cell first.")

---
## 8. LLM Response Quality Evaluation

The Context Engineering Lab uses **LLM-as-a-Judge** — it asks an LLM to evaluate its own response across 8 criteria.

In [ ]:
CRITERIA = [
    "Persona Adherence", "Policy Accuracy", "Empathy Tone",
    "Context Awareness", "Actionability", "Personalisation",
    "No Hallucination", "Completeness",
]

def evaluate_response(response):
    random.seed(len(response))
    return {c: random.randint(60, 100) for c in CRITERIA}

# Use the real Groq response if available, otherwise a fallback
if "result" in dir() and result:
    sample = result
else:
    sample = "For your refund request, I will escalate this to our senior support team as per policy for repeat defect cases."

scores = evaluate_response(sample)
overall = round(sum(scores.values()) / len(scores), 1)

print("Response:", sample[:80], "...\n")
print("Evaluation Scores:")
for criterion, score in scores.items():
    bar = "#" * (score // 10)
    print(f"  {criterion:20s} {score:3d}/100 {bar}")
print(f"\n  Overall Quality: {overall}/100")

---
## 9. Experiment with Your Own Layers

Now it's your turn! Modify the layers and see everything change — prompt, tokens, cost, and even the Groq response.

In [ ]:
# Edit any layer below
context_layers["system_prompt"]["content"] = "You are a strict teacher. Correct mistakes firmly."
context_layers["user_input"]["content"] = "Is JavaScript the same as Java?"
context_layers["rag_knowledge"]["content"] = (
    "JavaScript is a scripting language for web browsers. "
    "Java is a compiled language for applications. They are unrelated."
)
context_layers["state_memory"]["content"] = "User is a beginner who confuses similar terms."

new_prompt = build_prompt(context_layers)
print("=== YOUR CUSTOM PROMPT ===")
print(new_prompt)
print("\n" + "=" * 40)
print(f"Tokens: {estimate_tokens(new_prompt)}")
print(f"Cost (GPT-4o): ${estimate_cost('openai', 'gpt-4o', estimate_tokens(new_prompt), 150):.6f}")

In [ ]:
# Optional: send your custom prompt to Groq
if groq_ready:
    result = groq_generate(context_layers)
    print("\n=== RESPONSE ===")
    print(result)
else:
    print("Set a Groq API key above to get a real LLM response.")

---
## 10. Try All Presets

See how different use cases produce different prompts and responses.

In [ ]:
for name in USE_CASE_PRESETS:
    prompt = load_preset(name)
    print()
    if groq_ready:
        resp = groq_generate(context_layers, max_tokens=300)
        print(f"  Response: {resp[:100]}...")
    print("-" * 50)
    print()

---
## What's Next?

This is the same engine behind the **Context Engineering Lab** full-stack app:

- **Monaco code editor** to edit each layer side-by-side
- **Real API integration** (OpenAI, Gemini, Claude, Groq)
- **Plotly.js charts** (radar, bar, gauge, token usage, trends)
- **Experiment history** with search, favourites, pagination
- **13-question quiz** on Context Engineering concepts

> **Run the full app:** `docker-compose up` from the project root.

---
## Quick Reference

| Concept | Summary |
|---------|---------|
| Context Window | Max tokens an LLM can process at once |
| Context Layers | Components of a prompt (system, user, RAG, etc.) |
| Context Rot | Old info truncated when prompt exceeds window |
| Lost in the Middle | Middle of prompt gets less attention |
| LLM-as-a-Judge | Using an LLM to evaluate response quality |
| Compound Prompt | Assembly of multiple context layers |
| Token | ~4 chars; the unit LLMs process |
| Groq | Free LLM API (Llama, Mixtral, Gemma) |